[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_04/adding_reasoning.ipynb)

In [ ]:
# To run if the data is missing
import os
import shutil
from datasets import load_dataset

local_dir = "data/FinancialPhraseBank_augmented_judged"
repo_id = "lmassaron/FinancialPhraseBank_augmented_judged"

# If the corrupted/wrongly formatted directory exists, remove it
if os.path.exists(local_dir):
    print(f"Removing invalid directory: {local_dir}")
    shutil.rmtree(local_dir)

print(f"Downloading and formatting dataset...")
# 1. Load into memory/cache
dataset = load_dataset(repo_id)

# 2. Save it to disk in the exact Arrow format load_from_disk expects
dataset.save_to_disk(local_dir)
print(f"Dataset successfully saved to disk at {local_dir}!")

### Listing 4.11: Configuration Variables

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
DATASET_ID = "data/FinancialPhraseBank_augmented_judged"
OUTPUT_PATH = "data/FinancialPhraseBank_explained"
HF_OUTPUT_REPO = "username/FinancialPhraseBank_explained"
BATCH_SIZE = 8
MAX_NEW_TOKENS = 512

### Listing 4.12: Setting Prompt and Function for Sentiment Explanation

In [ ]:
EXPLAINER_SYSTEM_PROMPT = (
    "You are a senior financial analyst with deep expertise in equity markets, "
    "corporate finance, and macroeconomics. "
    "Your task is to explain, in 2-4 sentences, why a financial news headline "
    "carries a specific market sentiment. "
    "Focus strictly on the financial implications: how the news affects revenue, "
    "profitability, cash flow, investor confidence, or market positioning. "
    "Be concise and precise. Do not repeat the sentence verbatim."
)

def build_explain_prompt(sentence: str, sentiment: str) -> str:
    return (
        f'Financial news headline:\n"{sentence}"\n\n'
        f"This headline has been classified as **{sentiment}** sentiment "
        f"from a financial markets perspective.\n\n"
        f"Explain why, focusing on the financial implications for investors, "
        f"the company, or the broader market."
)


In [ ]:
# From the previous code listings
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import json
from tqdm import tqdm ###FIX
import re

QUANTIZATION_CONFIG = BitsAndBytesConfig( ###FIX
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def load_model(model_id: str):
    print(f"Loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        quantization_config=QUANTIZATION_CONFIG,
        device_map="auto",
    )
    model.eval()
    return tokenizer, model

tokenizer, model = load_model(MODEL_ID)

### Listing 4.13: Setting a Processing Function for Output Generation

In [ ]:
def process_dataset(tokenizer, model, dataset, limit=None):
    results = []
    sentences = dataset["sentence"]
    labels = dataset["label"]
    is_augmented = dataset["is_augmented"]
    judge_score = dataset["judge_score"]
    
    if limit:
        sentences = sentences[:limit]
        labels = labels[:limit]

    for start in tqdm(
        range(0, len(sentences), BATCH_SIZE), desc="Generating explanations"
    ):
        batch_sentences = sentences[start : start + BATCH_SIZE]
        batch_labels = labels[start : start + BATCH_SIZE]
        
        explain_prompts = [
            build_explain_prompt(s, LABEL_MAP[l])
            for s, l in zip(batch_sentences, batch_labels)
        ]

        explanations = generate_responses(tokenizer, model, EXPLAINER_SYSTEM_PROMPT, explain_prompts)

        for i, (sentence, label, explanation) in enumerate(
            zip(batch_sentences, batch_labels, explanations)
        ):
            idx = start + i
            results.append(
                {
                    "sentence": sentence,
                    "label": label,
                    "sentiment": LABEL_MAP[label],
                    "explanation": explanation.strip(),
                    "is_augmented": is_augmented[idx],
                    "judge_score": judge_score[idx],
                }
            )
    return results

### Listing 4.14: Executing the Processing Pipeline

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
from datasets import load_from_disk

In [ ]:
print(f"Loading augmented dataset from: {DATASET_ID}")
raw = load_from_disk(DATASET_ID)

output_splits = {}
for split_name, data in raw.items():
    print(f"\n── Processing split: {split_name} ──")
    results = process_dataset(tokenizer, model, data)
    output_splits[split_name] = Dataset.from_list(results)

output_ds = DatasetDict(output_splits)
output_ds.save_to_disk(OUTPUT_PATH)
print(f"\nDataset saved to: {OUTPUT_PATH}")

In [ ]:
ex = output_ds["train"][0]
print(f"Sentence  : {ex['sentence']}")
print(f"Sentiment : {ex['sentiment']}")
print(f"Augmented : {ex['is_augmented']}")
print(f"Explanation: {ex['explanation']}")